# ML-10 — Content Action Playbook

This notebook turns the validated ranking output into a practical, human-reviewed action queue. It does not auto-publish or claim that an action causes a ranking change.

## 1. Ranked actions + reason codes

The queue ranks pages by a blend of model probability and a transparent baseline. Reason codes explain why an item landed in the queue: model decline risk, visible demand, staleness, CTR review, engagement review, or thin-but-visible content. The first human pass should start with high-confidence pages that combine model risk and measurable demand.

In [1]:
from pathlib import Path
import sys
import pandas as pd

repo_candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/FlyRank-ML'), Path('/content/flyrank-ml')]
repo_root = next((p for p in repo_candidates if (p / 'work' / 'ml_track.py').exists()), Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from work.ml_track import (
    TARGET,
    ensure_dirs,
    load_analysis_frame,
    make_feature_matrix,
    run_artifacts,
    run_validation,
    write_json,
    write_paper_page,
)

ensure_dirs()
frame = load_analysis_frame()
print(f"Loaded {len(frame):,} rows across {frame['client_id'].nunique():,} client groups")
print(f"Observed snapshot-proxy base rate: {frame[TARGET].mean():.3f}")

artifacts = run_artifacts()
queue = artifacts['queue']
summary = artifacts['queue_summary']
display(queue[['rank', 'final_score', 'confidence', 'suggested_action', 'reason_codes', 'impressions_90d', 'sessions_90d']].head(20))
print('Reason-code counts:')
reason_counts = {}
for text in queue['reason_codes']:
    for reason in str(text).split('|'):
        reason_counts[reason] = reason_counts.get(reason, 0) + 1
display(pd.Series(reason_counts).sort_values(ascending=False).head(12).to_frame('rows'))

C:\Khalil\FlyRank-ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 111,133 rows across 49 client groups
Observed snapshot-proxy base rate: 0.644


,rank,final_score,confidence,suggested_action,reason_codes,impressions_90d,sessions_90d
0,1,89.108353,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,16607.0,147.0
1,2,88.415869,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,7508.0,189.0
2,3,88.047314,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,11671.0,143.0
3,4,87.972523,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,7895.0,299.0
4,5,87.913951,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,23037.0,165.0
5,6,87.902440,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,42453.0,155.0
6,7,87.851004,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,9990.0,252.0
7,8,87.633200,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,14291.0,254.0
8,9,87.483053,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,6333.0,200.0
9,10,87.464652,high,refresh_and_review_engagement,model_decline_risk|visible_model_opportunity|s...,12053.0,288.0


Reason-code counts:


,rows
general_review,38643
stale_visible_page,32304
visible_model_opportunity,32174
ctr_review_candidate,31000
model_decline_risk,25380
engagement_review_candidate,12551
thin_visible_page,25


## 2. Intended use and limits

**Intended use:** an SEO strategist or editor uses the ranked queue to choose which visible pages to inspect first, then checks the live page, search intent, seasonality, business priority, and recent changes.

**Limits:** the observed label is a current snapshot proxy; it is not a future treatment outcome. The queue is trained on anonymized data, does not contain page text or client context, and should not be used as an automatic rewrite, deletion, publishing, or budget decision.

In [2]:
assert queue['final_score'].between(0, 100).all()
assert queue['rank'].is_monotonic_increasing
print(f"Rows scored: {len(queue):,}")
print(f"Base rate used for context: {frame[TARGET].mean():.3f}")
print('Intended use check passed: reviewer decision-support only.')

Rows scored: 111,133
Base rate used for context: 0.644
Intended use check passed: reviewer decision-support only.


## 3. Human review + the no-go list

Before acting, a person must verify that the page is real, the page still serves the same intent, the demand is meaningful, the signal is not seasonal or caused by a site migration, and the recommended action is appropriate for the content owner.

**Never automate:** publishing or deleting content; changing canonical URLs or redirects; claiming causation; contacting clients; using private queries or client-identifying data; or treating a low-volume percentage swing as a business-impact verdict without context.

In [3]:
no_go_rules = [
    'no automatic publishing or deletion',
    'no canonical / redirect changes without a human owner',
    'no causal claims from this observational queue',
    'no private client data, raw queries, or credentials',
    'no action from a percentage swing without volume context',
]
print('\n'.join(f'- {rule}' for rule in no_go_rules))

- no automatic publishing or deletion
- no canonical / redirect changes without a human owner
- no causal claims from this observational queue
- no private client data, raw queries, or credentials
- no action from a percentage swing without volume context


## 4. Monitoring / retrain triggers

Re-run the queue on a regular cadence and investigate if the input mix changes, if the base rate shifts, if grouped holdout performance falls toward the base rate, or if the top reason codes no longer match editorial review. Retrain only after checking whether the label definition, data window, or tracking implementation changed.

In [4]:
monitor = {
    'cadence': 'monthly or after a material tracking/content-system change',
    'base_rate_shift': 'investigate when the observed decline rate moves materially from the receipt',
    'performance_trigger': 'investigate when grouped Precision@50 approaches the base rate',
    'drift_trigger': 'check feature distributions and missingness before retraining',
    'human_trigger': 'review if top reason codes repeatedly disagree with editors',
}
for key, value in monitor.items():
    print(f'{key}: {value}')

cadence: monthly or after a material tracking/content-system change
base_rate_shift: investigate when the observed decline rate moves materially from the receipt
performance_trigger: investigate when grouped Precision@50 approaches the base rate
drift_trigger: check feature distributions and missingness before retraining
human_trigger: review if top reason codes repeatedly disagree with editors


## 5. Exports for the paper

The notebook writes the complete queue to `work/outputs/ml10_action_queue.csv` (ignored from git because it is row-level data), while the compact JSON receipt and SVG figures remain safe to commit and embed in the paper.

In [5]:
write_json(repo_root / 'work' / 'outputs' / 'ml10_action_playbook_results.json', summary)
print('Wrote work/outputs/ml10_action_playbook_results.json')
print('Wrote work/outputs/ml10_action_queue.csv')
print('Figures: work/figures/action_mix.svg and work/figures/feature_importance.svg')

Wrote work/outputs/ml10_action_playbook_results.json
Wrote work/outputs/ml10_action_queue.csv
Figures: work/figures/action_mix.svg and work/figures/feature_importance.svg


## Self-check

- [x] Ranked actions have reason codes and human-readable limits
- [x] The no-go list prevents automatic or causal use
- [x] Monitoring and retrain triggers are explicit
- [x] Queue, JSON receipt, and figures are generated reproducibly
- [x] Claims remain public-safe and decision-support oriented